# Real-Time Violence Detection — Kaggle Preprocessing Pipeline\n\n**Bu notebook Kaggle üzerinde çalıştırılacaktır.**\n\n### Kaggle Kurulum Adımları:\n1. kaggle.com → \"Create\" → \"New Notebook\"\n2. Sağ panel → Settings → Accelerator: **GPU T4 x2**\n3. Sağ panel → \"Add Data\" → Arama: `real life violence situations dataset` → \"Add\"\n4. Bu notebook'un tüm hücrelerini sırasıyla Kaggle'a kopyalayın\n5. Çalıştırın → Output sekmesinden .npy dosyalarını indirin

In [ ]:
# ============================================================\n# Cell 1: Kütüphane Kurulumu ve Import\n# ============================================================\n!pip install ultralytics -q\n\nimport os\nimport cv2\nimport math\nimport numpy as np\nimport pandas as pd\nfrom glob import glob\nfrom tqdm import tqdm\nfrom ultralytics import YOLO\nfrom sklearn.model_selection import train_test_split\n\nprint(\"Kütüphaneler yüklendi.\")

In [ ]:
# ============================================================\n# Cell 2: Sabitler (decision_log.md D1–D20 kilitli değerler)\n# ============================================================\n\n# D1: FPS Standardization\nTARGET_FPS = 10\n\n# D2: Frame Preprocessing\nCLAHE_CLIP_LIMIT = 2.0\nCLAHE_TILE_GRID_SIZE = (8, 8)\nCLAHE_BRIGHTNESS_THRESHOLD = 50\nGAUSSIAN_KERNEL_SIZE = (3, 3)\nGAUSSIAN_SIGMA = 0\nRESIZE_DIM = (640, 640)\n\n# D3 / D12: Pose Model\nPOSE_MODEL_NAME = \"yolov8n-pose.pt\"\nNUM_KEYPOINTS = 17\nKEYPOINT_CONFIDENCE_THRESHOLD = 0.5\n\n# D4: Multi-Person\nMAX_PERSONS = 2\n\n# D13 / D14: Feature Vector\nFEATURE_DIM = 69\nSKELETON_DIM = 34\nTORSO_HEIGHT_EPSILON = 1e-6\n\n# D7: Sequence Window\nSEQUENCE_LENGTH = 30\nSLIDING_WINDOW_STRIDE = 15\n\n# D11: Motion Filter\nMOTION_FILTER_THETA = 0.05\n\n# D9: Split\nTRAIN_RATIO = 0.70\nVAL_RATIO = 0.15\nTEST_RATIO = 0.15\nSPLIT_RANDOM_STATE = 42\n\n# Labels\nVIOLENCE_LABEL = 1\nNONVIOLENCE_LABEL = 0\n\n# Kaggle Paths\nKAGGLE_INPUT_DIR = \"/kaggle/input/real-life-violence-situations-dataset\"\nOUTPUT_DIR = \"/kaggle/working\"\n\nprint(\"Sabitler tanımlandı.\")

In [ ]:
# ============================================================\n# Cell 3: Dataset Keşfi — Klasör yapısını doğrula\n# ============================================================\n\n# Kaggle dataset path'ini bul\nbase_path = KAGGLE_INPUT_DIR\nprint(f\"Input dizini: {base_path}\")\nprint(f\"İçerik: {os.listdir(base_path)}\")\n\n# Violence ve NonViolence klasörlerini bul\nfor root, dirs, files in os.walk(base_path):\n    if \"Violence\" in dirs or \"NonViolence\" in dirs:\n        VIOLENCE_DIR = os.path.join(root, \"Violence\")\n        NONVIOLENCE_DIR = os.path.join(root, \"NonViolence\")\n        break\n\nviolence_videos = sorted(glob(os.path.join(VIOLENCE_DIR, \"*\")))\nnonviolence_videos = sorted(glob(os.path.join(NONVIOLENCE_DIR, \"*\")))\n\nprint(f\"\\nViolence klasörü: {VIOLENCE_DIR}\")\nprint(f\"Violence video sayısı: {len(violence_videos)}\")\nprint(f\"NonViolence klasörü: {NONVIOLENCE_DIR}\")\nprint(f\"NonViolence video sayısı: {len(nonviolence_videos)}\")\nprint(f\"\\nÖrnek Violence: {violence_videos[0]}\")\nprint(f\"Örnek NonViolence: {nonviolence_videos[0]}\")

In [ ]:
# ============================================================\n# Cell 4: Preprocessing Fonksiyonları\n# ============================================================\n\ndef check_low_brightness(frame, threshold=CLAHE_BRIGHTNESS_THRESHOLD):\n    \"\"\"Frame'in ortalama parlaklığı threshold altındaysa True döner.\"\"\"\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n    return gray.mean() < threshold\n\n\ndef apply_clahe(frame):\n    \"\"\"LAB renk uzayında L kanalına CLAHE uygular.\"\"\"\n    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)\n    l_channel, a_channel, b_channel = cv2.split(lab)\n    clahe = cv2.createCLAHE(\n        clipLimit=CLAHE_CLIP_LIMIT,\n        tileGridSize=CLAHE_TILE_GRID_SIZE\n    )\n    l_channel = clahe.apply(l_channel)\n    lab = cv2.merge([l_channel, a_channel, b_channel])\n    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)\n\n\ndef preprocess_frame(frame):\n    \"\"\"\n    Locked preprocessing chain (decision_log.md D2):\n    CLAHE (conditional) → GaussianBlur 3×3 → Resize 640×640 → BGR→RGB\n    \"\"\"\n    # Step 1: Conditional CLAHE\n    if check_low_brightness(frame):\n        frame = apply_clahe(frame)\n\n    # Step 2: Gaussian Blur 3×3\n    frame = cv2.GaussianBlur(frame, GAUSSIAN_KERNEL_SIZE, GAUSSIAN_SIGMA)\n\n    # Step 3: Resize to 640×640\n    frame = cv2.resize(frame, RESIZE_DIM)\n\n    # Step 4: BGR → RGB\n    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n\n    return frame\n\n\nprint(\"Preprocessing fonksiyonları tanımlandı.\")

# ============================================================\n# Cell 5: YOLOv8n-Pose Model Yükleme + Pose Extraction\n# ============================================================\n\n# Model yükle (ilk çalıştırmada otomatik indirir)\npose_model = YOLO(POSE_MODEL_NAME)\nprint(f\"YOLOv8n-Pose modeli yüklendi: {POSE_MODEL_NAME}\")\n\n\ndef extract_pose(frame_rgb, model):\n    \"\"\"\n    YOLOv8n-Pose ile frame'den keypoint çıkar.\n    \n    Args:\n        frame_rgb: RGB formatında preprocessed frame (640x640)\n        model: YOLO pose modeli\n    \n    Returns:\n        List of dicts: her kişi için {keypoints, bbox, confidence}\n        Keypoints shape: (17, 3) — x, y, conf per keypoint\n    \"\"\"\n    results = model(frame_rgb, verbose=False)\n    persons = []\n    \n    if results[0].keypoints is not None and len(results[0].keypoints) > 0:\n        keypoints_data = results[0].keypoints.data.cpu().numpy()  # (N, 17, 3)\n        boxes_data = results[0].boxes.data.cpu().numpy()  # (N, 6) — x1,y1,x2,y2,conf,cls\n        \n        for i in range(len(keypoints_data)):\n            kps = keypoints_data[i]  # (17, 3)\n            box = boxes_data[i]  # (6,)\n            \n            # Bbox area for person selection\n            bbox_area = (box[2] - box[0]) * (box[3] - box[1])\n            # Bbox center\n            bbox_center_x = (box[0] + box[2]) / 2\n            bbox_center_y = (box[1] + box[3]) / 2\n            \n            persons.append({\n                'keypoints': kps,  # (17, 3) — x, y, conf\n                'bbox': box[:4],  # x1, y1, x2, y2\n                'bbox_area': bbox_area,\n                'bbox_center': (bbox_center_x, bbox_center_y),\n                'detection_conf': box[4]\n            })\n    \n    return persons\n\n\nprint(\"Pose extraction fonksiyonu tanımlandı.\")

In [ ]:
# ============================================================\n# Cell 6: Multi-Person Selection + Normalization + Feature Vector\n# ============================================================\n\ndef select_top2_persons(persons):\n    \"\"\"Top-2 kişi seçimi (bbox alanına göre) + X-axis sort (D5).\"\"\"\n    if len(persons) == 0:\n        return None, None\n    if len(persons) == 1:\n        return persons[0], None\n    sorted_by_area = sorted(persons, key=lambda p: p['bbox_area'], reverse=True)\n    top2 = sorted_by_area[:2]\n    top2_sorted = sorted(top2, key=lambda p: p['bbox_center'][0])\n    return top2_sorted[0], top2_sorted[1]\n\n\ndef filter_keypoints(keypoints):\n    \"\"\"Confidence < 0.5 olan keypoint'leri sıfırla (D12).\"\"\"\n    filtered = np.zeros((NUM_KEYPOINTS, 2), dtype=np.float32)\n    for i in range(NUM_KEYPOINTS):\n        if keypoints[i, 2] >= KEYPOINT_CONFIDENCE_THRESHOLD:\n            filtered[i, 0] = keypoints[i, 0]\n            filtered[i, 1] = keypoints[i, 1]\n    return filtered\n\n\ndef normalize_skeleton(keypoints_2d):\n    \"\"\"Hip centering + shoulder-hip scaling (D14).\"\"\"\n    hip_mid_x = (keypoints_2d[11, 0] + keypoints_2d[12, 0]) / 2\n    hip_mid_y = (keypoints_2d[11, 1] + keypoints_2d[12, 1]) / 2\n    \n    # Confidence masking: sıfır keypoint'leri hip_mid'e set et\n    for i in range(NUM_KEYPOINTS):\n        if keypoints_2d[i, 0] == 0.0 and keypoints_2d[i, 1] == 0.0:\n            keypoints_2d[i, 0] = hip_mid_x\n            keypoints_2d[i, 1] = hip_mid_y\n    \n    centered = keypoints_2d.copy()\n    centered[:, 0] -= hip_mid_x\n    centered[:, 1] -= hip_mid_y\n    \n    shoulder_mid_y = (keypoints_2d[5, 1] + keypoints_2d[6, 1]) / 2\n    torso_height = abs(hip_mid_y - shoulder_mid_y)\n    \n    if torso_height > TORSO_HEIGHT_EPSILON:\n        centered[:, 0] /= torso_height\n        centered[:, 1] /= torso_height\n    else:\n        return np.zeros((NUM_KEYPOINTS, 2), dtype=np.float32)\n    \n    return centered\n\n\ndef build_feature_vector(person1, person2, frame_width):\n    \"\"\"69-dim feature vector: [skeleton1(34)] + [skeleton2(34)] + [distance(1)].\"\"\"\n    if person1 is not None:\n        kps1 = filter_keypoints(person1['keypoints'])\n        skeleton1 = normalize_skeleton(kps1).flatten()\n    else:\n        skeleton1 = np.zeros(SKELETON_DIM, dtype=np.float32)\n    \n    if person2 is not None:\n        kps2 = filter_keypoints(person2['keypoints'])\n        skeleton2 = normalize_skeleton(kps2).flatten()\n    else:\n        skeleton2 = np.zeros(SKELETON_DIM, dtype=np.float32)\n    \n    if person1 is not None and person2 is not None:\n        raw_dist = math.dist(person1['bbox_center'], person2['bbox_center'])\n        norm_dist = raw_dist / frame_width if frame_width > 0 else 1.0\n    elif person1 is not None:\n        norm_dist = 1.0\n    else:\n        norm_dist = 0.0\n    \n    feature_vector = np.concatenate([\n        skeleton1, skeleton2,\n        np.array([norm_dist], dtype=np.float32)\n    ])\n    return feature_vector\n\n\nprint(\"Multi-person, normalization, feature vector fonksiyonları tanımlandı.\")

In [ ]:
# ============================================================\n# Cell 7: Tek Video İşleme Fonksiyonu (FPS Sampling + Pipeline)\n# ============================================================\n\ndef process_single_video(video_path, model):\n    \"\"\"\n    Tek bir videoyu baştan sona işler.\n    \n    Pipeline: FPS sampling → preprocess → pose → multi-person → normalize → 69-dim\n    \n    Returns:\n        np.array shape (n_frames, 69) or None if video is corrupt\n    \"\"\"\n    cap = cv2.VideoCapture(video_path)\n    \n    if not cap.isOpened():\n        return None, \"Video açılamadı\"\n    \n    native_fps = cap.get(cv2.CAP_PROP_FPS)\n    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    \n    if native_fps <= 0 or total_frames <= 0:\n        cap.release()\n        return None, f\"Geçersiz FPS={native_fps} veya frame_count={total_frames}\"\n    \n    # FPS sampling interval (D1: target 10 FPS)\n    frame_interval = max(1, round(native_fps / TARGET_FPS))\n    \n    feature_vectors = []\n    frame_idx = 0\n    \n    while True:\n        ret, frame = cap.read()\n        if not ret:\n            break\n        \n        # FPS sampling: sadece her frame_interval'inci kareyi al\n        if frame_idx % frame_interval == 0:\n            # Preprocessing chain (D2)\n            preprocessed = preprocess_frame(frame)\n            \n            # Pose extraction (D3)\n            persons = extract_pose(preprocessed, model)\n            \n            # Multi-person selection (D4/D5)\n            person1, person2 = select_top2_persons(persons)\n            \n            # Feature vector (D13)\n            fv = build_feature_vector(person1, person2, RESIZE_DIM[0])\n            feature_vectors.append(fv)\n        \n        frame_idx += 1\n    \n    cap.release()\n    \n    if len(feature_vectors) == 0:\n        return None, \"Hiç frame işlenemedi\"\n    \n    features = np.array(feature_vectors, dtype=np.float32)  # (n_frames, 69)\n    return features, None\n\n\nprint(\"Video işleme fonksiyonu tanımlandı.\")

In [ ]:
# ============================================================\n# Cell 8: Stratified Split (D9: 70/15/15)\n# ============================================================\n\nall_videos = violence_videos + nonviolence_videos\nall_labels = [VIOLENCE_LABEL] * len(violence_videos) + [NONVIOLENCE_LABEL] * len(nonviolence_videos)\n\nprint(f\"Toplam video: {len(all_videos)}\")\nprint(f\"Violence: {sum(1 for l in all_labels if l == 1)}\")\nprint(f\"NonViolence: {sum(1 for l in all_labels if l == 0)}\")\n\n# İlk test ayır (%15)\ntrain_val_files, test_files, train_val_labels, test_labels = train_test_split(\n    all_videos, all_labels,\n    test_size=TEST_RATIO,\n    stratify=all_labels,\n    random_state=SPLIT_RANDOM_STATE\n)\n\n# Sonra val ayır (kalan %85'ten ~%17.6 = toplamın %15'i)\nval_ratio_adjusted = VAL_RATIO / (1 - TEST_RATIO)\ntrain_files, val_files, train_labels, val_labels = train_test_split(\n    train_val_files, train_val_labels,\n    test_size=val_ratio_adjusted,\n    stratify=train_val_labels,\n    random_state=SPLIT_RANDOM_STATE\n)\n\nprint(f\"\\nSplit sonuçları:\")\nprint(f\"Train: {len(train_files)} video ({sum(1 for l in train_labels if l==1)} V / {sum(1 for l in train_labels if l==0)} NV)\")\nprint(f\"Val:   {len(val_files)} video ({sum(1 for l in val_labels if l==1)} V / {sum(1 for l in val_labels if l==0)} NV)\")\nprint(f\"Test:  {len(test_files)} video ({sum(1 for l in test_labels if l==1)} V / {sum(1 for l in test_labels if l==0)} NV)\")\n\n# Split bilgisini dict olarak sakla\nsplit_map = {}\nfor f, l in zip(train_files, train_labels):\n    split_map[f] = ('train', l)\nfor f, l in zip(val_files, val_labels):\n    split_map[f] = ('val', l)\nfor f, l in zip(test_files, test_labels):\n    split_map[f] = ('test', l)\n\n# CSV kaydet\nfor split_name, files, labels in [('train', train_files, train_labels),\n                                    ('val', val_files, val_labels),\n                                    ('test', test_files, test_labels)]:\n    df = pd.DataFrame({\n        'filepath': files,\n        'label': labels,\n        'split': split_name\n    })\n    csv_path = os.path.join(OUTPUT_DIR, f\"{split_name}.csv\")\n    df.to_csv(csv_path, index=False)\n    print(f\"Kaydedildi: {csv_path}\")

In [ ]:
# ============================================================\n# Cell 9: Tüm Videoları İşle → Per-Video .npy\n# ============================================================\n\n# Output klasörlerini oluştur\nfor split in ['train', 'val', 'test']:\n    for label in ['violence', 'nonviolence']:\n        os.makedirs(os.path.join(OUTPUT_DIR, 'features', split, label), exist_ok=True)\n\n# İşleme log'u\nprocess_log = []\nskipped_videos = []\n\nprint(f\"Toplam {len(all_videos)} video işlenecek...\")\nprint(\"=\"*60)\n\nfor video_path in tqdm(all_videos, desc=\"Video İşleme\"):\n    split_name, label = split_map[video_path]\n    \n    # Dosya adını al\n    video_filename = os.path.splitext(os.path.basename(video_path))[0]\n    label_str = 'violence' if label == VIOLENCE_LABEL else 'nonviolence'\n    \n    # .npy kayıt yolu\n    npy_filename = f\"{split_name}_{label_str}_{video_filename}.npy\"\n    npy_path = os.path.join(OUTPUT_DIR, 'features', split_name, label_str, npy_filename)\n    \n    # Video işle\n    features, error = process_single_video(video_path, pose_model)\n    \n    if features is None:\n        skipped_videos.append({'video': video_path, 'error': error})\n        continue\n    \n    # Kaydet\n    np.save(npy_path, features)\n    process_log.append({\n        'video': video_path,\n        'split': split_name,\n        'label': label_str,\n        'n_frames': features.shape[0],\n        'shape': str(features.shape),\n        'npy_path': npy_path\n    })\n\nprint(f\"\\n{'='*60}\")\nprint(f\"İşlenen video: {len(process_log)}\")\nprint(f\"Atlanan video: {len(skipped_videos)}\")\n\nif skipped_videos:\n    print(\"\\nAtlanan videolar:\")\n    for s in skipped_videos[:10]:\n        print(f\"  {s['video']}: {s['error']}\")

In [ ]:
# ============================================================\n# Cell 10: Doğrulama — Rastgele .npy dosyalarını kontrol et\n# ============================================================\nimport random\n\n# Rastgele 5 dosya kontrol et\nnpy_files = glob(os.path.join(OUTPUT_DIR, 'features', '**', '*.npy'), recursive=True)\nprint(f\"Toplam .npy dosyası: {len(npy_files)}\")\n\nsample_files = random.sample(npy_files, min(5, len(npy_files)))\n\nprint(\"\\nRastgele 5 dosya kontrolü:\")\nprint(\"-\" * 60)\nfor f in sample_files:\n    arr = np.load(f)\n    has_nan = np.isnan(arr).any()\n    has_inf = np.isinf(arr).any()\n    print(f\"  {os.path.basename(f)}\")\n    print(f\"    Shape: {arr.shape} | dtype: {arr.dtype}\")\n    print(f\"    Min: {arr.min():.3f} | Max: {arr.max():.3f}\")\n    print(f\"    NaN: {has_nan} | Inf: {has_inf}\")\n    if has_nan or has_inf:\n        print(f\"    ⚠️ UYARI: Geçersiz değer tespit edildi!\")\n    print()

In [ ]:
# ============================================================\n# Cell 11: Sliding Window + Motion Filter → Sequences\n# ============================================================\n\ndef create_sliding_windows(features, window_size=SEQUENCE_LENGTH, stride=SLIDING_WINDOW_STRIDE):\n    \"\"\"(n_frames, 69) → list of (30, 69) windows.\"\"\"\n    n_frames = features.shape[0]\n    windows = []\n    \n    if n_frames < window_size:\n        # Kısa video: sıfır padding\n        padded = np.zeros((window_size, FEATURE_DIM), dtype=np.float32)\n        padded[:n_frames] = features\n        windows.append(padded)\n    else:\n        for start in range(0, n_frames - window_size + 1, stride):\n            window = features[start:start + window_size]\n            windows.append(window)\n    \n    return windows\n\n\ndef compute_motion_score(window):\n    \"\"\"Penceredeki ardışık frame çiftleri arası ortalama L2 mesafesi.\"\"\"\n    diffs = np.diff(window, axis=0)  # (29, 69)\n    frame_distances = np.linalg.norm(diffs, axis=1)  # (29,)\n    return frame_distances.mean()\n\n\ndef apply_motion_filter(windows, theta=MOTION_FILTER_THETA):\n    \"\"\"Düşük hareketli Violence pencerelerini ele (D11).\"\"\"\n    filtered = []\n    for w in windows:\n        score = compute_motion_score(w)\n        if score >= theta:\n            filtered.append(w)\n    return filtered\n\n\n# Her split için sekanslar oluştur\nfor split in ['train', 'val', 'test']:\n    os.makedirs(os.path.join(OUTPUT_DIR, 'sequences', split), exist_ok=True)\n    \n    violence_sequences = []\n    nonviolence_sequences = []\n    \n    # Violence .npy dosyaları\n    v_files = glob(os.path.join(OUTPUT_DIR, 'features', split, 'violence', '*.npy'))\n    for f in v_files:\n        features = np.load(f)\n        windows = create_sliding_windows(features)\n        # Motion filter sadece Violence'a uygulanır (D11)\n        filtered_windows = apply_motion_filter(windows)\n        violence_sequences.extend(filtered_windows)\n    \n    # NonViolence .npy dosyaları (filtresiz)\n    nv_files = glob(os.path.join(OUTPUT_DIR, 'features', split, 'nonviolence', '*.npy'))\n    for f in nv_files:\n        features = np.load(f)\n        windows = create_sliding_windows(features)\n        nonviolence_sequences.extend(windows)\n    \n    print(f\"\\n{split.upper()} Split:\")\n    print(f\"  Violence sekans (filter sonrası): {len(violence_sequences)}\")\n    print(f\"  NonViolence sekans: {len(nonviolence_sequences)}\")\n    \n    # Sınıf dengesi: undersampling (train için)\n    if split == 'train' and len(nonviolence_sequences) > len(violence_sequences):\n        random.seed(SPLIT_RANDOM_STATE)\n        nonviolence_sequences = random.sample(nonviolence_sequences, len(violence_sequences))\n        print(f\"  NonViolence undersampled → {len(nonviolence_sequences)}\")\n    \n    # Kaydet\n    X = np.array(violence_sequences + nonviolence_sequences, dtype=np.float32)\n    y = np.array(\n        [VIOLENCE_LABEL] * len(violence_sequences) + \n        [NONVIOLENCE_LABEL] * len(nonviolence_sequences),\n        dtype=np.float32\n    )\n    \n    np.save(os.path.join(OUTPUT_DIR, 'sequences', split, f'X_{split}.npy'), X)\n    np.save(os.path.join(OUTPUT_DIR, 'sequences', split, f'y_{split}.npy'), y)\n    \n    print(f\"  Final: X_{split}.npy shape={X.shape}, y_{split}.npy shape={y.shape}\")\n    print(f\"  Kaydedildi: {OUTPUT_DIR}/sequences/{split}/\")

In [ ]:
# ============================================================\n# Cell 12: Final Doğrulama + Özet\n# ============================================================\n\nprint(\"=\"*60)\nprint(\"FINAL DOĞRULAMA\")\nprint(\"=\"*60)\n\nfor split in ['train', 'val', 'test']:\n    X = np.load(os.path.join(OUTPUT_DIR, 'sequences', split, f'X_{split}.npy'))\n    y = np.load(os.path.join(OUTPUT_DIR, 'sequences', split, f'y_{split}.npy'))\n    \n    n_violence = int((y == 1).sum())\n    n_nonviolence = int((y == 0).sum())\n    \n    print(f\"\\n{split.upper()}:\")\n    print(f\"  X shape: {X.shape} | dtype: {X.dtype}\")\n    print(f\"  y shape: {y.shape} | dtype: {y.dtype}\")\n    print(f\"  Violence: {n_violence} | NonViolence: {n_nonviolence}\")\n    print(f\"  NaN: {np.isnan(X).any()} | Inf: {np.isinf(X).any()}\")\n    print(f\"  Min: {X.min():.3f} | Max: {X.max():.3f}\")\n\nprint(f\"\\n{'='*60}\")\nprint(\"Output dosyaları (Kaggle Output sekmesinden indirilecek):\")\nprint(\"-\"*60)\nfor root, dirs, files in os.walk(os.path.join(OUTPUT_DIR, 'sequences')):\n    for f in files:\n        full = os.path.join(root, f)\n        size_mb = os.path.getsize(full) / (1024*1024)\n        print(f\"  {os.path.relpath(full, OUTPUT_DIR)} — {size_mb:.1f} MB\")\n\nprint(f\"\\nSplit CSV'leri:\")\nfor split in ['train', 'val', 'test']:\n    print(f\"  {split}.csv\")\n\nprint(f\"\\n✅ Preprocessing tamamlandı! Output sekmesinden dosyaları indirin.\")

In [ ]:
# ============================================================\n# Cell 13: Preprocessing Log Kaydet\n# ============================================================\n\n# İşleme log'unu CSV olarak kaydet\nif process_log:\n    log_df = pd.DataFrame(process_log)\n    log_df.to_csv(os.path.join(OUTPUT_DIR, 'preprocessing_log.csv'), index=False)\n    print(f\"Preprocessing log kaydedildi: {len(log_df)} video\")\n\nif skipped_videos:\n    skip_df = pd.DataFrame(skipped_videos)\n    skip_df.to_csv(os.path.join(OUTPUT_DIR, 'skipped_videos.csv'), index=False)\n    print(f\"Atlanan videolar log'u kaydedildi: {len(skip_df)} video\")\n\nprint(\"\\nTüm çıktılar /kaggle/working/ altında.\")\nprint(\"Kaggle Notebook'un sağ tarafındaki 'Output' sekmesinden indirebilirsiniz.\")